# ఖర్చు క్లెయిం విశ్లేషణ

ఈ నోట్‌బుక్ లోకల్ రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను ప్రాసెస్ చేయడానికి ప్లగిన్లను ఉపయోగించే ఏజెంట్లు ఎలా సృష్టించాలో, ఖర్చు క్లెయిం ఇమెయిల్‌ను ఉత్పత్తి చేయడాన్ని, మరియు ప్యే చార్ట్ ఉపయోగించి ఖర్చు డేటాను దృష్టిగతం చేయడాన్ని చూపిస్తుంది. ఏజెంట్లు కార్యాచరణ సందర్భానుసారం ఫంక్షన్లను డైనమిక్గా ఎంచుకుంటాయి.

దశలు:
1. OCR ఏజెంట్ లోకల్ రసీదు చిత్రాన్ని ప్రాసెస్ చేసి ప్రయాణ ఖర్చు డేటాను తీసుకుంటుంది.
2. ఇమెయిల్ ఏజెంట్ ఖర్చు క్లెయిం ఇమెయిల్‌ను సృష్టిస్తుంది.

### ప్రయాణ ఖర్చు సన్నివేశం యొక్క ఉదాహరణ:
మీరు మరో నగరంలో వ్యాపార సమావేశానికి వెళుతున్న ఉద్యోగి అని ఊహించండి. మీ కంపెనీ అన్ని సమర్థనీయమైన ప్రయాణ సంబంధిత ఖర్చులను తిరిగి చెల్లించే విధానం కలదు. ఇక్కడ సాధ్యమైన ప్రయాణ ఖర్చుల విభజన ఉంది:
- ట్రాన్స్‌పోర్టేషన్:
మీ ఇంటి నగరంతో గమ్యస్థాన నగరానికి గుండ్రటి ప్రయాణానికి విమానభటన.
ఎయిర్‌పోర్ట్‌కు వెళ్లడం మరియు వచ్చేదానికి టాక్సీ లేదా రైడ్-హెయిలింగ్ సేవలు.
గమ్యస్థాన నగరంలో లోకల్ ట్రాన్స్‌పోర్టేషన్ (పబ్లిక్ ట్రాన్సిట్, రెంటల్ కార్లు లేదా టాక్సీలు వంటి).

- వసతి:
సమావేశ స్థలానికి దగ్గరగా మధ్యస్థాయిలో వ్యాపార హోటల్‌లో మూడు రాత్రులు ఉంటూ.

- భోజనాలు:
కంపెనీ యొక్క రోజువారీ ధోరణి ప్రాతిపదికగా ఉదయం భోజనం, మధ్యాహ్న భోజనం మరియు రాత్రి భోజనానికి దినచర్య భోజన అనుమతి.

- వివిధ ఖర్చులు:
ఎయిర్‌పోర్ట్ వద్ద పార్కింగ్ ఫీజులు.
హోటల్‌లో ఇంటర్నెట్ యాక్సెస్ చార్జీలు.
సూచనలు లేదా చిన్న సర్వీసు చార్జీలు.

- డాక్యుమెంటేషన్:
మీరు అన్ని రసీదులు (ఫ్లైట్లు, టాక్సీలు, హోటల్, భోజనాలు, మొదలైనవి) మరియు పూర్తయిన ఖర్చు నివేదికను రీయింబర్స్మెంట్ కోసం సమర్పిస్తారు.


## అవసరమైన లైబ్రరీలను దిగుమతి చేసుకోండి

నోట్బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మాడ్యూల్స్‌ని దిగుమతి చేసుకోండి.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## ఖర్చు మోడల్స్‌ను నిర్వచించండి

 వ్యక్తిగత ఖర్చుల కోసం ఒక Pydantic మోడల్ మరియు వినియోగదారు ప్రశ్నను నిర్మిత ఖర్చు డేటాగా మార్చేందుకు ExpenseFormatter క్లాస్‌ను రూపొందించండి.

 ప్రతి ఖర్చు ఈ ఫార్మాట్‌లో ప్రాతినిథ్యం వహిస్తుంది:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## పరికరాలను నిర్వచించడం - ఇమెయిల్ సృష్టించడం

ఖర్చుల దావాకు ఇమెయిల్‌ను తయారుచేసేందుకు ఒక పరికరము ఫంక్షన్ సృష్టించండి.
- ఈ పరికరం Microsoft Agent Framework నుండి `@tool` డెకరేటర్‌ను ఉపయోగిస్తుంది.
- ఇది ఖర్చుల మొత్తం మొత్తాన్ని లెక్కించి, వివరాలను ఇమెల్ శరీరంగా ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను వెలికి తీయడానికి టూల్  

రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను వెలికి తీయడానికి ఒక టూల్ ఫంక్షన్ సృష్టించండి.  
- ఈ టూల్ Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ నుండి `@tool` డెకొరేటర్‌ను ఉపయోగిస్తుంది.  
- ఇది రసీదు చిత్రాన్ని చదవడం, దాన్ని base64 గా కోడ్ చేయడం, మరియు ఏజెంట్ విశ్లేషించడానికి డేటా URIని ফেরుస్తుంది.  


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

Define the agents and wire them into a sequential workflow using `WorkflowBuilder`.
- The OCR agent extracts structured expense data from the receipt image using the `load_receipt_image` tool.
- The Email agent takes the extracted data and generates a professional expense claim email using the `generate_expense_email` tool.
- `WorkflowBuilder` with `add_edge` creates a sequential pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## ప్రధాన ఫังก్షన్

వరుస వర్క్‌ఫ్లోని నిర్మించి, రసీత చిత్రం ప్రాసెస్ చేయడానికి మరియు ఖర్చు అభ్యర్థన ఇమెయిల్ సృష్టించడానికి దాన్ని నడపండి.


> **గమనిక:** ఈ పనిముట్లు ప్రస్తుతానికి రశీదు చిత్రాన్ని base64 టెక్స్ట్ గా పంపుతుంది, ఇది ఎక్కువ చాట్ మోడల్స్ (gpt-4.1-mini సహా) చిత్రంగా పరిగణించవు.
> ఇది మోడల్ కాంటెక్ట్ విండోను మించిపోయే అవకాశం కూడా ఉంది. OCRని Azure AI Vision (లేదా మరో OCR టూల్) తో నడిపించి, మాత్రమే తీయబడిన పాఠ్యాన్ని పంపడం మంచిది, లేక చిత్రం `image_url` సందేశంగా పంపేలా మళ్ళించండి.
> మీరు మాత్రమే కాంటెక్ట్ లో లోపాలు కాకుండా ఉండాలనుకుంటే, చిన్న రశీదు చిత్రాన్ని లేదా పెద్ద కాంటెక్ట్ విండో గల మోడల్‌ని ప్రయత్నించండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
